# Vector Store Ingestion

Transform the semantically rich chunks from Stage 3 into a queryable **Vector Database**. This forms the core of the RAG pipeline's retrieval system.

### Objectives
1. **Vector Store Initialization**: Set up a persistent ChromaDB instance.
2. **Base Embedding Generation**: Generate baseline embeddings for the text chunks.
3. **Hybrid Indexing**: Include metadata (Citations, Hierarchy, Filenames) for faceted search.
4. **Retrieval Benchmarking**: Perform qualitative spot-checks to establish a baseline.

## 1. Setup

Import necessary libraries and define paths for the input processed chunks and the persistent vector store.

In [1]:
import pandas as pd
import json
from pathlib import Path
from tqdm.auto import tqdm
import chromadb
from sentence_transformers import SentenceTransformer

# Input and Output Paths
DATA_PATH = Path('../data/processed/san_diego_code_chunks.jsonl')
DB_PATH = Path('../data/vector_store/san_diego_code_baseline')

# Load data
df = pd.read_json(DATA_PATH, lines=True)

print(f"Loaded {len(df)} chunk records.")
df.head()

/Users/Aresh/Desktop/AAI 590/sd-land-use-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/Aresh/Desktop/AAI 590/sd-land-use-rag/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Loaded 52384 chunk records.


,text,metadata,source
0,Article 3: Land Development Procedures,"{'chapter': 'chapter_11', 'filename': 'Ch11Art...",Ch11Art03Division02.pdf
1,Division 2: Rules for Calculation and Measurem...,"{'chapter': 'chapter_11', 'filename': 'Ch11Art...",Ch11Art03Division02.pdf
2,§113.0201,"{'chapter': 'chapter_11', 'filename': 'Ch11Art...",Ch11Art03Division02.pdf
3,Purpose of Rules for Calculation and Measurement,"{'chapter': 'chapter_11', 'filename': 'Ch11Art...",Ch11Art03Division02.pdf
4,Article 3: Land Development Procedures > Purpo...,"{'chapter': 'chapter_11', 'filename': 'Ch11Art...",Ch11Art03Division02.pdf


## 2. Model & Vector Store Initialization

### 2.1 Load Embedding Model

Use `sentence-transformers/all-MiniLM-L6-v2` as the standard baseline model for embedding.

In [2]:
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

# Initialize the SentenceTransformer model
model = SentenceTransformer(MODEL_NAME)

# Test the model
if model:
    sample_embedding = model.encode(["This is a test sentence."])
    print(f"Embedding dimension: {sample_embedding.shape[1]}")

Embedding dimension: 384


### 2.2 Initialize ChromaDB

Create a persistent ChromaDB client and a collection to store document embeddings.

In [3]:
# Initialize a PersistentClient for ChromaDB pointing to DB_PATH
chroma_client = chromadb.PersistentClient(DB_PATH)

# Reset collection if exists
try:
    chroma_client.delete_collection("san_diego_code")
    print("Deleted existing collection.")
except Exception:
    pass

# Create a new collection named 'san_diego_code'
collection = chroma_client.create_collection("san_diego_code", metadata={"hnsw:space": "cosine"})

if collection:
    print(f"Collection '{collection.name}' initialized. Current count: {collection.count()}")

Deleted existing collection.
Collection 'san_diego_code' initialized. Current count: 0


## 3. Batch Ingestion Pipeline

To manage memory, embed and ingest documents in batches. For each chunk, prepare the `ids`, `documents` (text), and `metadatas`.

In [4]:
BATCH_SIZE = 500
total_records = len(df)

def prepare_metadata(metadata_dict):
    """Flatten dicts and handle lists so ChromaDB can index them properly."""
    cleaned_meta = {}
    cleaned_meta['chapter'] = metadata_dict.get('chapter', '')
    cleaned_meta['filename'] = metadata_dict.get('filename', '')
    
    hierarchy = metadata_dict.get('hierarchy', {})
    cleaned_meta['article'] = hierarchy.get('article', '')
    cleaned_meta['division'] = hierarchy.get('division', '')
    cleaned_meta['section'] = hierarchy.get('section', '')
    
    citations = metadata_dict.get('citations', [])
    if citations:
        cleaned_meta['citations'] = citations
    
    return cleaned_meta

print("Starting ingestion...")
for i in tqdm(range(0, total_records, BATCH_SIZE)):
    batch_df = df.iloc[i:i+BATCH_SIZE]
    
    texts = batch_df['text'].tolist()
    metadatas = [prepare_metadata(m) for m in batch_df['metadata']]
    
    # Create unique IDs like chunk_0, chunk_1
    ids = [f"chunk_{idx}" for idx in batch_df.index]
    
    # Generate embeddings for this batch of texts using `model.encode()`
    embeddings = model.encode(texts)
    
    # Add IDs, embeddings, metadatas, and documents to the ChromaDB collection
    if collection and embeddings is not None:
        collection.add(ids=ids, embeddings = embeddings.tolist(), metadatas=metadatas, documents=texts)

if collection:
    print(f"Ingestion complete. Total items in collection: {collection.count()}")

Starting ingestion...


100%|██████████| 105/105 [00:43<00:00,  2.42it/s]

Ingestion complete. Total items in collection: 52384


## 4. Retrieval Benchmarking

Test the baseline vector store by performing semantic search on common zoning questions.

In [5]:
def query_vector_store(query_text, n_results=5, where_filter=None):
    """
    Helper function to query the collection and print results.
    """
    if not model or not collection:
        print("Model or collection not initialized.")
        return
        
    # Generate the embedding for the query_text
    query_embedding = model.encode(query_text).tolist()
    
    # Query the collection using `collection.query()`
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results, where=where_filter)
    
    if results:
        print(f"\n--- Query: '{query_text}' ---")
        if where_filter:
            print(f"Filters: {where_filter}")
        
        for i in range(len(results['documents'][0])):
            dist = results['distances'][0][i] if 'distances' in results else 'N/A'
            meta = results['metadatas'][0][i]
            text = results['documents'][0][i]
            
            print(f"\n[Result {i+1} | Distance: {dist:.4f}]")
            print(f"Source: {meta.get('section', 'Unknown Section')} ({meta.get('filename')})")
            if meta.get('citations'):
                print(f"Citations referenced: {meta.get('citations')}")
            print(f"Text:\n{text[:300]}...")

### 4.1 Semantic Search Tests

Test standard RAG questions.

In [6]:
# Example Query
query_vector_store("What are the height limits for residential zones?")

# Add a query testing specific building codes
query_vector_store("What is the definition of a front yard setback?")


--- Query: 'What are the height limits for residential zones?' ---

[Result 1 | Distance: 0.2546]
Source: (ii) (Ch13Art01Division04.pdf)
Text:
Article 1: Base Zones > Division 4: Residential Base Zones (Added 12-9-1997 by O-18451 N.S.; effective 1-1-2000.) > (ii): (iii) Complies with all applicable structure height limits in...

[Result 2 | Distance: 0.2749]
Source: Ch. Art. Div. 4 1 13 (Ch13Art01Division04.pdf)
Citations referenced: ['113.0270']
Text:
Article 1: Base Zones > Division 4: Residential Base Zones (Added 12-9-1997 by O-18451 N.S.; effective 1-1-2000.) > Ch. Art. Div. 4 1 13: base zone maximum structure height shall be 30 feet, which shall be determined in accordance with Section 113.0270(a)(4)(D)....

[Result 3 | Distance: 0.2824]
Source: Ground-floor Height (Ch13Art01Division04.pdf)
Text:
Article 1: Base Zones > Division 4: Residential Base Zones (Added 12-9-1997 by O-18451 N.S.; effective 1-1-2000.) > Ground-floor Height: Ground-floor height requirements apply to struct

### 4.2 Metadata-Driven Retrieval (Hybrid Search)

Test search using ChromaDB's filtering capabilities to restrict search scope.

In [7]:
# Example Filtered Query: Search for setback rules strictly within Article 3
query_vector_store(
    query_text="setback regulations", 
    n_results=3, 
    where_filter={"article": "Article 3: Land Development Procedures"}
)

# Try a query where you filter by a specific citation using the 'citations' metadata field
query_vector_store(query_text="How is building height measured?", n_results=3, where_filter={"citations":{"$contains":"113.0201"}})



--- Query: 'setback regulations' ---
Filters: {'article': 'Article 3: Land Development Procedures'}

[Result 1 | Distance: 0.2621]
Source: Setbacks (Ch11Art03Division02.pdf)
Text:
Setbacks...

[Result 2 | Distance: 0.3661]
Source: §113.0252 Measuring Setbacks (Ch11Art03Division02.pdf)
Citations referenced: ['113.0252']
Text:
§113.0252 Measuring Setbacks...

[Result 3 | Distance: 0.3921]
Source: Side Setback (Ch11Art03Division02.pdf)
Text:
Side Setback...

--- Query: 'How is building height measured?' ---
Filters: {'citations': {'$contains': '113.0201'}}

[Result 1 | Distance: 0.9252]
Source:  (Ch11Art03Division02.pdf)
Citations referenced: ['113.0201']
Text:
§113.0201...


## 5. Export Baseline Evaluation

Run standard questions against the baseline model, extract the results, and export them to `baseline_eval.csv`.

In [8]:
import csv

evaluation_queries = [
    "What are the height limits for residential zones?",
    "What is the definition of a front yard setback?",
    "Where is multiple dwelling unit development permitted?",
    "What are the parking requirements for commercial zones?",
    "How is building height measured?"
]

eval_results = []
for q in evaluation_queries:
    q_emb = model.encode(q).tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=5)
    for i in range(len(res['documents'][0])):
        eval_results.append({
            'Query': q,
            'Rank': i+1,
            'Distance': res['distances'][0][i],
            'Source': f"{res['metadatas'][0][i].get('section', 'Unknown Section')} ({res['metadatas'][0][i].get('filename')})",
            'Citations': res['metadatas'][0][i].get('citations', ''),
            'Text_Snippet': res['documents'][0][i][:300]
        })

eval_df = pd.DataFrame(eval_results)
eval_df.to_csv('../data/vector_store/baseline_eval.csv', index=False)
print(f"Exported {len(eval_results)} evaluation logs to baseline_eval.csv")
eval_df.head()

Exported 25 evaluation logs to baseline_eval.csv


,Query,Rank,Distance,Source,Citations,Text_Snippet
0,What are the height limits for residential zones?,1,0.254595,(ii) (Ch13Art01Division04.pdf),,Article 1: Base Zones > Division 4: Residentia...
1,What are the height limits for residential zones?,2,0.274896,Ch. Art. Div. 4 1 13 (Ch13Art01Division04.pdf),[113.0270],Article 1: Base Zones > Division 4: Residentia...
2,What are the height limits for residential zones?,3,0.282355,Ground-floor Height (Ch13Art01Division04.pdf),,Article 1: Base Zones > Division 4: Residentia...
3,What are the height limits for residential zones?,4,0.283600,§131.0540 Maximum Permitted Residential Densit...,,Article 1: Base Zones > §131.0540 Maximum Perm...
4,What are the height limits for residential zones?,5,0.285731,§131.0540 Maximum Permitted Residential Densit...,,Article 1: Base Zones > §131.0540 Maximum Perm...
